# Notebook 4: Regression Analysis
**Course:** Economic and Social Statistics (AST3336) — Year 3, Semester 2
**Institution:** University of Rwanda, College of Business and Economics
**Group:** Group 1 — AHS 2024: Poverty & Demographics

---

## Objective
Estimate regression models to identify demographic and livelihood characteristics associated with poverty:

1. **Logit Model** — Binary poverty status (`poor`) as dependent variable
2. **Probit Model** — Same specification, alternative estimator
3. **Marginal Effects** — Average marginal effects for both models
4. **OLS Baseline Model** — Welfare quintile (`welfare_quintile`) as dependent variable
5. **Model comparison & interpretation**

### Model Specification
$$poor_i = f(head\_sex_i,\ head\_age_i,\ head\_age\_sq_i,\ hh\_size_i,\ dep\_ratio_i,\ livelihood\_ag_i,\ land\_ha\_log_i,\ d\_south_i,\ d\_west_i,\ d\_north_i,\ d\_east_i)$$

**Input:** `data/ahs2024_analysis.csv`
**Output:** Tables → `output/tables/`, Figures → `output/figures/`

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Logit, Probit
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH   = r'..\data\ahs2024_analysis.csv'
TABLE_PATH  = r'..\output\tables'
FIGURE_PATH = r'..\output\figures'

df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]} households, {df.shape[1]} variables')
print(f'\nKey variables:')
print(df[['poor','welfare_quintile','head_sex','head_age','hh_size',
          'dep_ratio','livelihood_ag','land_ha_log']].describe().round(3))

## 2. Prepare Regression Dataset

In [ ]:
# Define regressors
regressors = [
    'head_sex',      # Sex of HH head (1=Male, 0=Female)
    'head_age',      # Age of HH head
    'head_age_sq',   # Age squared (non-linear effect)
    'hh_size',       # Household size
    'dep_ratio',     # Dependency ratio
    'livelihood_ag', # Main livelihood (1=Agriculture)
    'land_ha_log',   # Log of agricultural land
    'd_south',       # Province dummies (Kigali = reference)
    'd_west',
    'd_north',
    'd_east'
]

# Dependent variables
dep_binary   = 'poor'
dep_ols      = 'welfare_quintile'

# Drop rows with missing values in any regression variable
reg_vars = [dep_binary, dep_ols] + regressors
df_reg = df[reg_vars].dropna().copy()

print(f'Full dataset:      {len(df)} households')
print(f'Regression sample: {len(df_reg)} households')
print(f'Dropped (missing): {len(df) - len(df_reg)} households')
print(f'\nPoverty rate in regression sample: {df_reg["poor"].mean()*100:.2f}%')

In [ ]:
# Prepare X and y matrices
X = df_reg[regressors].astype(float)
X_const = sm.add_constant(X)  # Add intercept
y_logit  = df_reg[dep_binary].astype(float)
y_ols    = df_reg[dep_ols].astype(float)

print('X matrix shape:', X_const.shape)
print('y (poor) shape:', y_logit.shape)
print('y (quintile) shape:', y_ols.shape)
print('\nCorrelation of regressors with poverty:')
print(X.corrwith(y_logit).round(4).sort_values())

## 3. Logit Model

In [ ]:
# -------------------------------------------------------
# Estimate Logit Model
# -------------------------------------------------------
logit_model  = Logit(y_logit, X_const)
logit_result = logit_model.fit(maxiter=200, disp=True)

print(logit_result.summary2())

In [ ]:
# Odds Ratios from Logit
logit_or = pd.DataFrame({
    'Coefficient'  : logit_result.params,
    'Odds Ratio'   : np.exp(logit_result.params),
    'Std. Error'   : logit_result.bse,
    'z-statistic'  : logit_result.tvalues,
    'p-value'      : logit_result.pvalues,
    'CI Lower (OR)': np.exp(logit_result.conf_int()[0]),
    'CI Upper (OR)': np.exp(logit_result.conf_int()[1])
}).round(4)

# Add significance stars
def stars(p):
    if p < 0.01:  return '***'
    elif p < 0.05: return '**'
    elif p < 0.1:  return '*'
    else:          return ''

logit_or['Sig.'] = logit_or['p-value'].apply(stars)

print('=== LOGIT MODEL — COEFFICIENTS & ODDS RATIOS ===')
print(logit_or.to_string())

logit_or.to_csv(os.path.join(TABLE_PATH, 'table7_logit_results.csv'))
print('\nLogit results saved ✓')

## 4. Probit Model

In [ ]:
# -------------------------------------------------------
# Estimate Probit Model
# -------------------------------------------------------
probit_model  = Probit(y_logit, X_const)
probit_result = probit_model.fit(maxiter=200, disp=True)

print(probit_result.summary2())

In [ ]:
# Probit coefficients table
probit_table = pd.DataFrame({
    'Coefficient': probit_result.params,
    'Std. Error' : probit_result.bse,
    'z-statistic': probit_result.tvalues,
    'p-value'    : probit_result.pvalues,
    'CI Lower'   : probit_result.conf_int()[0],
    'CI Upper'   : probit_result.conf_int()[1]
}).round(4)

probit_table['Sig.'] = probit_table['p-value'].apply(stars)

print('=== PROBIT MODEL — COEFFICIENTS ===')
print(probit_table.to_string())

probit_table.to_csv(os.path.join(TABLE_PATH, 'table8_probit_results.csv'))
print('\nProbit results saved ✓')

## 5. Marginal Effects

In [ ]:
# -------------------------------------------------------
# Average Marginal Effects — Logit
# -------------------------------------------------------
logit_me = logit_result.get_margeff(at='mean')
print('=== LOGIT — MARGINAL EFFECTS AT MEAN ===')
print(logit_me.summary())

In [ ]:
# -------------------------------------------------------
# Average Marginal Effects — Probit
# -------------------------------------------------------
probit_me = probit_result.get_margeff(at='mean')
print('=== PROBIT — MARGINAL EFFECTS AT MEAN ===')
print(probit_me.summary())

In [ ]:
# -------------------------------------------------------
# Combined marginal effects table (Logit vs Probit)
# -------------------------------------------------------
me_logit_df = pd.DataFrame({
    'Logit ME'    : logit_me.margeff,
    'Logit SE'    : logit_me.margeff_se,
    'Logit p-val' : logit_me.pvalues,
    'Logit Sig.'  : pd.Series(logit_me.pvalues).apply(stars).values
}, index=logit_me.summary_frame().index)

me_probit_df = pd.DataFrame({
    'Probit ME'   : probit_me.margeff,
    'Probit SE'   : probit_me.margeff_se,
    'Probit p-val': probit_me.pvalues,
    'Probit Sig.' : pd.Series(probit_me.pvalues).apply(stars).values
}, index=probit_me.summary_frame().index)

me_combined = pd.concat([me_logit_df, me_probit_df], axis=1).round(4)

print('=== COMBINED MARGINAL EFFECTS — LOGIT vs PROBIT ===')
print(me_combined.to_string())

me_combined.to_csv(os.path.join(TABLE_PATH, 'table9_marginal_effects.csv'))
print('\nMarginal effects saved ✓')

## 6. OLS Baseline Model

In [ ]:
# -------------------------------------------------------
# OLS Model — welfare quintile as dependent variable
# -------------------------------------------------------
ols_model  = sm.OLS(y_ols, X_const)
ols_result = ols_model.fit(cov_type='HC3')  # Robust standard errors

print(ols_result.summary())

In [ ]:
# OLS results table
ols_table = pd.DataFrame({
    'Coefficient': ols_result.params,
    'Std. Error' : ols_result.bse,
    't-statistic': ols_result.tvalues,
    'p-value'    : ols_result.pvalues,
    'CI Lower'   : ols_result.conf_int()[0],
    'CI Upper'   : ols_result.conf_int()[1]
}).round(4)

ols_table['Sig.'] = ols_table['p-value'].apply(stars)

print('=== OLS MODEL — WELFARE QUINTILE ===')
print(f'R-squared: {ols_result.rsquared:.4f}')
print(f'Adj. R-squared: {ols_result.rsquared_adj:.4f}')
print(f'F-statistic: {ols_result.fvalue:.4f} (p={ols_result.f_pvalue:.4f})')
print(f'Observations: {int(ols_result.nobs)}')
print()
print(ols_table.to_string())

ols_table.to_csv(os.path.join(TABLE_PATH, 'table10_ols_results.csv'))
print('\nOLS results saved ✓')

## 7. Model Fit Statistics

In [ ]:
# -------------------------------------------------------
# Model fit comparison table
# -------------------------------------------------------
fit_stats = pd.DataFrame({
    'Model'         : ['Logit', 'Probit', 'OLS'],
    'Dep. Variable' : ['Poor (binary)', 'Poor (binary)', 'Welfare Quintile'],
    'N'             : [int(logit_result.nobs), int(probit_result.nobs), int(ols_result.nobs)],
    'Log-Likelihood': [round(logit_result.llf, 2), round(probit_result.llf, 2), round(ols_result.llf, 2)],
    'AIC'           : [round(logit_result.aic, 2), round(probit_result.aic, 2), round(ols_result.aic, 2)],
    'BIC'           : [round(logit_result.bic, 2), round(probit_result.bic, 2), round(ols_result.bic, 2)],
    'Pseudo/R²'     : [
        round(logit_result.prsquared, 4),
        round(probit_result.prsquared, 4),
        round(ols_result.rsquared, 4)
    ]
})

print('=== MODEL FIT STATISTICS ===')
print(fit_stats.to_string(index=False))

fit_stats.to_csv(os.path.join(TABLE_PATH, 'table11_model_fit.csv'), index=False)
print('\nModel fit table saved ✓')

## 8. Figures

In [ ]:
# -------------------------------------------------------
# Figure 6: Marginal effects plot (Logit)
# -------------------------------------------------------
me_df = logit_me.summary_frame().copy()
me_df = me_df.drop(index='const', errors='ignore')

# Variable labels for plot
var_labels = {
    'head_sex'     : 'Male HH Head',
    'head_age'     : 'Age of HH Head',
    'head_age_sq'  : 'Age² of HH Head',
    'hh_size'      : 'Household Size',
    'dep_ratio'    : 'Dependency Ratio',
    'livelihood_ag': 'Ag. Livelihood',
    'land_ha_log'  : 'Land (log ha)',
    'd_south'      : 'South Province',
    'd_west'       : 'West Province',
    'd_north'      : 'North Province',
    'd_east'       : 'East Province'
}
me_df.index = [var_labels.get(i, i) for i in me_df.index]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#d32f2f' if v > 0 else '#1976d2' for v in me_df['dy/dx']]
bars = ax.barh(me_df.index, me_df['dy/dx'], color=colors,
               xerr=me_df['Std. Err.'], capsize=3,
               error_kw={'linewidth': 1}, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Average Marginal Effects on Poverty Probability\nLogit Model — AHS 2024',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Marginal Effect on P(Poor=1)', fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_PATH, 'fig6_marginal_effects.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 6 saved ✓')

In [ ]:
# -------------------------------------------------------
# Figure 7: OLS coefficients plot
# -------------------------------------------------------
ols_plot = ols_table.drop(index='const', errors='ignore').copy()
ols_plot.index = [var_labels.get(i, i) for i in ols_plot.index]

fig, ax = plt.subplots(figsize=(9, 6))
colors_ols = ['#388e3c' if v > 0 else '#f57c00' for v in ols_plot['Coefficient']]
ax.barh(ols_plot.index, ols_plot['Coefficient'],
        xerr=ols_plot['Std. Error'], capsize=3,
        color=colors_ols, edgecolor='white',
        error_kw={'linewidth': 1})
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('OLS Coefficients — Effect on Welfare Quintile\nAHS 2024',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Coefficient (Welfare Quintile)', fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_PATH, 'fig7_ols_coefficients.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 7 saved ✓')

In [ ]:
# -------------------------------------------------------
# Figure 8: Predicted probability of poverty by key variables
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Predicted Probability of Poverty — Logit Model\nAHS 2024',
             fontsize=13, fontweight='bold')

# Predicted probability by HH size
df_reg['pred_prob'] = logit_result.predict(X_const)
pred_size = df_reg.groupby('hh_size')['pred_prob'].mean()
axes[0].plot(pred_size.index, pred_size.values, 'o-', color='#1976d2', linewidth=2)
axes[0].set_title('By Household Size')
axes[0].set_xlabel('Household Size (members)')
axes[0].set_ylabel('Predicted P(Poor)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Predicted probability by age group
df['pred_prob_full'] = np.nan
df.loc[df_reg.index, 'pred_prob_full'] = logit_result.predict(X_const).values
df['head_age_group'] = pd.cut(
    df['head_age'], bins=[0,30,45,60,120],
    labels=['15-30','31-45','46-60','61+']
)
pred_age = df.groupby('head_age_group', observed=True)['pred_prob_full'].mean()
axes[1].bar(pred_age.index.astype(str), pred_age.values,
            color='#f57c00', edgecolor='white')
axes[1].set_title('By Age Group of HH Head')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Predicted P(Poor)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_PATH, 'fig8_predicted_probabilities.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 8 saved ✓')

## 9. Interpretation Summary

In [ ]:
# -------------------------------------------------------
# Print interpretation guide
# -------------------------------------------------------
print('=== INTERPRETATION GUIDE ===')
print()
print('--- LOGIT MODEL ---')
print('Coefficients: Log-odds of being poor')
print('Odds Ratios:  OR > 1 = increases odds of poverty')
print('              OR < 1 = decreases odds of poverty')
print()
print('--- PROBIT MODEL ---')
print('Coefficients: Change in z-score of poverty probability')
print('Sign same as Logit but magnitude smaller (~0.6x)')
print()
print('--- MARGINAL EFFECTS ---')
print('dy/dx: Change in P(poor=1) for unit change in X')
print('e.g. ME = -0.05 means variable reduces poverty probability by 5pp')
print()
print('--- OLS MODEL ---')
print('Coefficient: Change in welfare quintile for unit change in X')
print('Positive = improves welfare quintile (moves toward richest)')
print('Negative = worsens welfare quintile (moves toward poorest)')
print()

print('=== KEY SIGNIFICANT FINDINGS ===')
sig_logit = logit_or[logit_or['p-value'] < 0.05].drop(index='const', errors='ignore')
print('\nSignificant variables in Logit (p < 0.05):')
for var in sig_logit.index:
    coef = sig_logit.loc[var, 'Coefficient']
    OR   = sig_logit.loc[var, 'Odds Ratio']
    p    = sig_logit.loc[var, 'p-value']
    direction = 'INCREASES' if coef > 0 else 'DECREASES'
    print(f'  {var}: {direction} poverty odds (OR={OR:.3f}, p={p:.4f})')

sig_ols = ols_table[ols_table['p-value'] < 0.05].drop(index='const', errors='ignore')
print('\nSignificant variables in OLS (p < 0.05):')
for var in sig_ols.index:
    coef = sig_ols.loc[var, 'Coefficient']
    p    = sig_ols.loc[var, 'p-value']
    direction = 'IMPROVES' if coef > 0 else 'WORSENS'
    print(f'  {var}: {direction} welfare quintile (β={coef:.4f}, p={p:.4f})')

## 10. Final Output Summary

In [ ]:
print('=== ALL REGRESSION OUTPUTS ===')
print('\nTABLES saved:')
for f in sorted(os.listdir(TABLE_PATH)):
    if f.endswith('.csv'):
        print(f'  ✓ {f}')

print('\nFIGURES saved:')
for f in sorted(os.listdir(FIGURE_PATH)):
    if f.endswith('.png'):
        print(f'  ✓ {f}')

print('\n✓ All 4 notebooks complete!')
print('✓ Ready to write Chapter 3 & Chapter 4 in LaTeX')

## Summary of Models

| Model | DV | Estimator | Key Output |
|-------|----|-----------|------------|
| Logit | `poor` (binary) | Maximum Likelihood | Coefficients, Odds Ratios |
| Probit | `poor` (binary) | Maximum Likelihood | Coefficients |
| Marginal Effects | `poor` (binary) | At mean | dy/dx |
| OLS | `welfare_quintile` | OLS + HC3 SE | Coefficients, R² |

**Regressors (all models):**
- `head_sex` — sex of household head
- `head_age` + `head_age_sq` — age of head (with non-linear effect)
- `hh_size` — household size
- `dep_ratio` — dependency ratio
- `livelihood_ag` — main livelihood activity
- `land_ha_log` — log of agricultural land
- `d_south`, `d_west`, `d_north`, `d_east` — province dummies (Kigali = reference)

**Next:** Write Chapter 3 (Methodology) and Chapter 4 (Results & Discussion) in LaTeX